This notebook is served as the code of Appendix C.4 "Cascading bandit".

1. Experiment Name: Ablation study of varying $k$ under ascending-order cascading feedback;
2. Dataset: synthetic;
3. Range: $[0,0.1]$;
4. Description: In this notebook, we vary $k$ ($k \in \{5,10,15,20,25\}$, fixed $m=30$) in the ascending-order cascading feedback setting.

In [1]:
import random
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

GLOBAL_SEED = 2026
# -------- Bandit Class --------
class PerturbedBernoulliBandit:
    def __init__(self, mu, noise_scale):
        self.base_mu = np.clip(mu, 0, 0.1)
        self.noise_scale = noise_scale
    
    def pull(self):
        perturbed_mu = np.clip(
            self.base_mu + np.random.normal(0, self.noise_scale),
            0, 0.1
        )
        return np.random.binomial(1, perturbed_mu)

# -------- our proposed CMOSS algorithm --------  
class CMOSS:
    def __init__(self, m, k, delta):
        self.m = m
        self.k = k
        self.delta = delta
        self.T_i = np.zeros(m)          
        self.hat_mu = np.ones(m)       
    
    def select_arms(self):
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, 1e-8, self.T_i)  
        log_input = 1 / (self.delta * T_safe)
        log_input = np.clip(log_input, 1.0, 1e100)        
        rho = np.sqrt(np.log(log_input) / T_safe)
        rho = np.nan_to_num(rho, nan=0.0, posinf=0.0, neginf=0.0)
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)       
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- CUCB algorithm --------
class CUCB:
    def __init__(self, m, k):
        self.m = m
        self.k = k
        self.T_i = np.zeros(m)       
        self.hat_mu = np.ones(m)     
        self.t = 0                   
    
    def select_arms(self):
        self.t += 1                   
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, np.inf, self.T_i)  
        rho = np.sqrt(3 * np.log(self.t) / (2 * T_safe))    
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)        
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):    
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- ablation study of varying k under ascending-order cascading feedback --------    
class ExperimentRunner:
    def __init__(self, noise_scale):
        self.noise_scale = noise_scale
        self.data_dir = "results"
        os.makedirs(self.data_dir, exist_ok=True)

        self.param_grid = {
            'CMOSS': {'k': [5, 10, 15, 20, 25], 'delta': [0.00001]},
            'CUCB': {'k': [5, 10, 15, 20, 25]},
        }

        self.datasets = self._generate_datasets()
        
    def _generate_datasets(self):
        return {
            'synthetic_30arms': {
                'm': 30,
                'mus': np.random.uniform(0, 0.1, 30).tolist()
            }
        }


    def _gen_params(self, algo):
        if algo == 'CMOSS':
            return [{'k': k, 'delta': d} for k in self.param_grid[algo]['k'] for d in self.param_grid[algo]['delta']]
        if algo == 'CUCB':
            return [{'k': k} for k in self.param_grid[algo]['k']]
    
    def _init_algo(self, algo, m, params):
        if algo == 'CMOSS':
            return CMOSS(m=m, k=params['k'], delta=params['delta'])
        elif algo == 'CUCB':
            return CUCB(m=m, k=params['k'])
        
    def _cascading_reward(self, arms, bandits):
        """Compute expected reward under cascading model."""
        no_click_prob = 1.0
        for arm_idx in arms:
            no_click_prob *= (1 - bandits[arm_idx].base_mu)
        return 1 - no_click_prob

    def run(self, n_sim, T):
        conv_data= []

        for dataset_name, dataset_info in self.datasets.items():
            m, mus = dataset_info['m'], dataset_info['mus']
            sorted_mus = sorted(mus, reverse=True)
            for algo in ['CMOSS', 'CUCB']:
                for params in self._gen_params(algo):
                    k = params['k']
                    if k > m:
                        print(f"Skipping: k={k} > m={m}")
                        continue

                    best_exp = 1 - np.prod([1 - mu for mu in sorted_mus[:k]])

                    for sim_id in tqdm(range(n_sim), desc=f"{dataset_name}-{algo}-k{k}"):
                        sim_seed = GLOBAL_SEED + sim_id * 1000 + hash((algo, k)) % 10_000_000
                        random.seed(sim_seed)
                        np.random.seed(sim_seed)
                            
                        bandits = [PerturbedBernoulliBandit(mu, self.noise_scale) for mu in mus]
                        agent = self._init_algo(algo, len(mus), params)

                        cumulative_time = 0.0
                        current_regret = 0.0

                        regrets = []
                        times = []

                        for t in range(T):
                            step_start = time.perf_counter()
                            arms = agent.select_arms()[::-1]  # reverse order for ascending cascade

                            # Simulate cascading click feedback
                            rewards = [0] * k
                            clicked_index = -1
                            for pos, arm_idx in enumerate(arms):
                                if bandits[arm_idx].pull() == 1:
                                    rewards[pos] = 1
                                    clicked_index = pos
                                    break  # Cascade ends after first click

                            observed_rewards = [0] * k
                            for i in range(k):
                                if i == clicked_index:
                                    observed_rewards[i] = 1
                                    break
                                else:
                                    observed_rewards[i] = 0
                            agent.update(arms[:clicked_index+1] if clicked_index >= 0 else arms, 
                                            observed_rewards[:clicked_index+1] if clicked_index >= 0 else observed_rewards)

                            # Regret computation
                            current_reward = 1 - np.prod([1 - bandits[i].base_mu for i in arms])
                            current_regret += best_exp - current_reward
                            regrets.append(current_regret)

                            step_time = time.perf_counter() - step_start
                            cumulative_time += step_time
                            times.append(cumulative_time)

                            # Store final data
                            conv_data.append({
                                "k": k,
                                "m": m,
                                "algorithm": algo,
                                "dataset": dataset_name,
                                "sim_id": sim_id,
                                "t": t + 1,
                                'noise': self.noise_scale,
                                "regret": current_regret,
                                "cumulative_time": cumulative_time
                            })
        return pd.DataFrame(conv_data)

    def save_data(self, conv_df):
        conv_df.to_csv(f"{self.data_dir}/l_conv_synthetic_varying_k.csv", index=False)

        
if __name__ == "__main__":
    exp = ExperimentRunner(noise_scale=0)
    n_sim, T = 10, 100000
    conv_df= exp.run(n_sim, T)
    exp.save_data(conv_df)

synthetic_30arms-CUCB-k25: 100%|██████████| 10/10 [05:57<00:00, 35.71s/it]


In [2]:
def process_results(input_path, output_path, target_steps):
    df = pd.read_csv(input_path)

    filtered = df[df['t'].isin(target_steps)]

    grouped = filtered.groupby(['algorithm', 'm','k','t']).agg({
        'regret': 'mean',
        'cumulative_time': 'mean'
    }).reset_index()
    
    result = grouped.melt(
        id_vars=['algorithm', 'm','k','t'],
        value_vars=['regret', 'cumulative_time'],
        var_name='metric',
        value_name='average_value'
    )
    
    result.to_csv(output_path, index=False)
    print(f"Processed results saved to {output_path}")

if __name__ == "__main__":
    config = {
        'input_file': 'results\l_conv_synthetic_varying_k.csv',
        'output_file': 'results\l_summary_synthetic_varying_k.csv',
        'target_steps': [100000]
    }
    
    process_results(
        input_path=config['input_file'],
        output_path=config['output_file'],
        target_steps=config['target_steps']
    )

Processed results saved to results\l_summary_synthetic_varying_k.csv
